# CS F425 Deep Learning Project
## QLoRA Fine-Tuning of Phi-2 for Structured Data Analysis

Fine-tunes `microsoft/phi-2` (4-bit QLoRA) to act as an AI agent over `sales_data.csv`.  
Given a natural language query, the model outputs:
```json
{"actions": ["filter_data(column='year', value=2022)", "aggregate_sum(column='revenue')"], "answer": 52345678.12}
```

**Runtime target**: ≤6 hours on Colab free-tier T4 GPU.

## Cell 1 — Install Packages

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

DEPS_SENTINEL = Path("/tmp/.csf425_phi2_deps_ready")
PACKAGES = [
    "transformers>=4.41.0",
    "peft>=0.10.0",
    "trl>=0.9.0",
    "accelerate>=0.29.3",
    "datasets>=2.19.0",
    "bitsandbytes>=0.44.0",
    "einops",
]

if not DEPS_SENTINEL.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *PACKAGES])
    DEPS_SENTINEL.write_text("ok")
    print("Dependencies installed or upgraded. Restarting the Colab runtime once...")
    os.kill(os.getpid(), 9)

print("Dependencies already prepared for this runtime.")


In [ ]:
# Cell 1 restarts the Colab runtime once after upgrading packages.
# After the reconnect, continue from the next cell.


## Cell 2 — Imports & GPU Check

In [ ]:
import json
import random
import re

import pandas as pd
import torch
from datasets import Dataset
from packaging import version
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

try:
    import trl
except Exception as exc:
    raise RuntimeError(
        "Failed to import trl. Re-run Cell 1 and let Colab restart once."
    ) from exc

if not hasattr(trl, "SFTTrainer"):
    raise RuntimeError(
        "trl.SFTTrainer is unavailable. Re-run Cell 1 and let Colab restart once."
    )

SFTTrainer = trl.SFTTrainer
SFTConfig = getattr(trl, "SFTConfig", TrainingArguments)
_USE_SFTCONFIG = hasattr(trl, "SFTConfig")

import transformers as _tf

if version.parse(_tf.__version__) < version.parse("4.41.0"):
    raise RuntimeError(
        f"transformers=={_tf.__version__} is too old for this notebook. "
        "Re-run Cell 1 and let Colab restart once."
    )

print(f"trl         : {trl.__version__}  (SFTConfig available: {_USE_SFTCONFIG})")
print(f"transformers: {_tf.__version__}")

assert torch.cuda.is_available(), "No GPU. In Colab, switch to a T4 GPU runtime."
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Cell 3 — Mount Google Drive

Saves adapter weights and checkpoints to Drive so they persist across Colab sessions.

**One-time setup**: Upload these files to `MyDrive/CS_F425_Project/`:
- `sales_data.csv`
- `agent_trajectories_2k.json`
- `tool_executor.py`
- `run_pipeline.py`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, glob as _glob
DRIVE_DIR = "/content/drive/MyDrive/CS_F425_Project"
ADAPTER_DIR = f"{DRIVE_DIR}/phi2-agent-adapter"
CKPT_DIR = f"{DRIVE_DIR}/phi2-agent-qlora"

os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# Make tool_executor.py importable
sys.path.insert(0, DRIVE_DIR)


def detect_training_state():
    """Detect what checkpoints are available and decide the execution path.

    Returns (state, best_path) where state is one of:
      TRAINING_COMPLETE  - final adapter saved (skip training, go to inference)
      EPOCH_CHECKPOINT   - HF Trainer checkpoint exists (resume training)
      TIMED_CHECKPOINT   - timed adapter-only snapshot (skip training, inference-capable)
      FRESH              - nothing found (train from scratch)

    Checkpoints are sorted by their numeric suffix (highest number wins),
    so deleting earlier checkpoints to save space will not cause issues.
    """
    # 1. Final adapter saved by Cell 13?
    if os.path.isfile(f"{ADAPTER_DIR}/adapter_config.json"):
        return "TRAINING_COMPLETE", ADAPTER_DIR

    # 2. Trainer epoch checkpoints (contain full optimizer state)?
    epoch_ckpts = _glob.glob(f"{CKPT_DIR}/checkpoint-*")
    if epoch_ckpts:
        epoch_ckpts.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
        return "EPOCH_CHECKPOINT", epoch_ckpts[-1]

    # 3. Timed adapter-only snapshots?
    timed_ckpts = _glob.glob(f"{ADAPTER_DIR}/timed_ckpt_step_*")
    if timed_ckpts:
        timed_ckpts.sort(key=lambda p: int(p.rsplit("_", 1)[-1]))
        return "TIMED_CHECKPOINT", timed_ckpts[-1]

    return "FRESH", None


TRAINING_STATE, BEST_CKPT_PATH = detect_training_state()
SKIP_TRAINING = TRAINING_STATE in ("TRAINING_COMPLETE", "TIMED_CHECKPOINT")

print(f"Drive dir       : {DRIVE_DIR}")
print(f"Contents        : {os.listdir(DRIVE_DIR)}")
print(f"Training state  : {TRAINING_STATE}")
print(f"Best checkpoint : {BEST_CKPT_PATH}")
print(f"Skip training   : {SKIP_TRAINING}")

## Cell 4 — Load Data

In [ ]:
df = pd.read_csv(f"{DRIVE_DIR}/sales_data.csv")
print(f"sales_data shape: {df.shape}")
print(df.dtypes)
print(df.head(3))

with open(f"{DRIVE_DIR}/agent_trajectories_2k.json") as f:
    trajectories = json.load(f)
print(f"\nTrajectories loaded: {len(trajectories)}")
print("Sample entry:", json.dumps(trajectories[0], indent=2))

## Cell 5 — Pre-compute Answers

The training data has `query` + `actions` but **no answers**.  
We execute each action sequence against `sales_data.csv` using `ToolExecutor` to obtain the gold answer.

We also extend `parse_agent_action` to handle `aggregate_mean`, `aggregate_count`, and string filter values that the original `run_pipeline.py` misses.

In [ ]:
from numbers import Integral, Real

from tool_executor import ToolExecutor


def _parse_numeric(val_str):
    """Convert a string to int or float as appropriate."""
    return float(val_str) if '.' in val_str else int(val_str)


def parse_agent_action(action_str):
    """Parse one string-form action into ToolExecutor format."""
    if action_str.startswith("filter_data"):
        # Try numeric value first (int or float, possibly negative)
        m = re.search(r"column='([^']+)',\s*value=(-?[\d]+(?:\.[\d]+)?)", action_str)
        if m:
            return {
                "tool": "filter",
                "args": {"column": m.group(1), "op": "==", "value": _parse_numeric(m.group(2))},
            }

        # Try string value
        m = re.search(r"column='([^']+)',\s*value='([^']+)'", action_str)
        if m:
            return {
                "tool": "filter",
                "args": {"column": m.group(1), "op": "==", "value": m.group(2)},
            }

    elif action_str.startswith("group_by"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "groupby", "args": {"column": m.group(1)}}

    elif action_str.startswith("aggregate_sum"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "sum"}}

    elif action_str.startswith("aggregate_mean"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "mean"}}

    elif action_str.startswith("aggregate_count"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "count"}}

    elif action_str.startswith("sort_by"):
        m = re.search(r"column='([^']+)',\s*order='([^']+)'", action_str)
        if m:
            return {
                "tool": "sort",
                "args": {"column": m.group(1), "ascending": m.group(2) != "desc"},
            }

    elif action_str.startswith("top_k"):
        m = re.search(r"k=(\d+)", action_str)
        if m:
            return {"tool": "topk", "args": {"k": int(m.group(1))}}

    return None


def clean_scalar(value):
    if hasattr(value, "item"):
        try:
            value = value.item()
        except Exception:
            pass

    if isinstance(value, bool):
        return bool(value)
    if isinstance(value, Integral):
        return int(value)
    if isinstance(value, Real):
        return round(float(value), 4)
    return value


def result_to_python(actions, result_df):
    """Convert ToolExecutor output into the JSON shape used for supervision."""
    if result_df is None or (hasattr(result_df, "empty") and result_df.empty):
        return None

    action_names = [action.split("(")[0] for action in actions]

    # Single scalar result
    if result_df.shape == (1, 1):
        return clean_scalar(result_df.iloc[0, 0])

    # group_by + aggregate (without sort/topk) → key-value dict
    has_groupby = any(a.startswith("group_by") for a in action_names)
    has_sort_or_topk = any(
        a.startswith("sort_by") or a.startswith("top_k") for a in action_names
    )
    if result_df.shape[1] == 2 and has_groupby and not has_sort_or_topk:
        key_col, value_col = result_df.columns
        return {
            str(row[key_col]): clean_scalar(row[value_col])
            for _, row in result_df.iterrows()
        }

    # Default: list of dicts (for sorted/topk results or multi-column outputs)
    return [
        {str(key): clean_scalar(value) for key, value in row.items()}
        for row in result_df.to_dict(orient="records")
    ]


def compute_answer(actions, df):
    """Parse and execute a list of action strings."""
    parsed = []
    for action in actions:
        try:
            parsed_action = parse_agent_action(action)
        except Exception:
            parsed_action = None

        if parsed_action is not None:
            parsed.append(parsed_action)

    if not parsed:
        return None

    try:
        result = ToolExecutor(df.copy()).execute(parsed)
        return result_to_python(actions, result)
    except Exception:
        return None


training_data = []
skipped = 0

for entry in trajectories:
    answer = compute_answer(entry["actions"], df)
    if answer is None:
        skipped += 1
        continue

    output_json = json.dumps(
        {"actions": entry["actions"], "answer": answer},
        ensure_ascii=False,
    )
    training_data.append({"query": entry["query"], "output_json": output_json})

print(f"Valid examples : {len(training_data)} / {len(trajectories)}")
print(f"Skipped        : {skipped}")
print("\nSample:")
print(json.dumps(json.loads(training_data[0]["output_json"]), indent=2, ensure_ascii=False))

## Cell 6 — Schema String & Prompt Template

The prompt follows the same `### Task / ### Schema / ### Question / ### Answer` pattern as Lab02.  
The schema string is fixed — it is identical at training and inference time.

In [ ]:
SCHEMA = """Table: sales_data
Columns: date (date), year (int), month (int), city (str), region (str),
         product (str), category (str), revenue (float), units_sold (int),
         cost (float), profit (float)

Available actions (use exactly this syntax):
  filter_data(column='col', value=val)
  group_by(column='col')
  aggregate_sum(column='col')
  aggregate_mean(column='col')
  aggregate_count(column='col')
  sort_by(column='col', order='asc'|'desc')
  top_k(k=N)"""

PROMPT_TEMPLATE = """### Task
Analyze the sales data and answer the query.
Output ONLY a JSON object with an \"actions\" list and an \"answer\" field. No other text.

### Schema
{schema}

### Question
{question}

### Answer
"""

MAX_SEQ_LENGTH = 384


def make_prompt(question: str) -> str:
    return PROMPT_TEMPLATE.format(schema=SCHEMA, question=question)


EOS = None  # Set properly in Cell 8 after tokenizer loads


def format_example(row, eos_token=None):
    eos = eos_token or EOS
    if eos is None:
        raise ValueError("EOS token not set. Run Cell 8 (tokenizer) before Cell 9.")
    return {"text": make_prompt(row["query"]) + row["output_json"] + eos}


print(make_prompt(training_data[0]["query"]))
print(f"Configured max sequence length: {MAX_SEQ_LENGTH}")

## Cell 7 — Split Dataset

In [ ]:
random.seed(42)
random.shuffle(training_data)

TRAIN_SIZE = min(1800, int(len(training_data) * 0.9))
VALID_SIZE = 200

raw_train = training_data[:TRAIN_SIZE]
raw_valid = training_data[TRAIN_SIZE : TRAIN_SIZE + VALID_SIZE]

print(f"Train : {len(raw_train)}")
print(f"Valid : {len(raw_valid)}")

## Cell 8 — Load Tokenizer

In [ ]:
MODEL_NAME = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
EOS = tokenizer.eos_token

print(f"EOS token: {EOS!r}")

sample_text = make_prompt(raw_train[0]["query"]) + raw_train[0]["output_json"] + EOS
n_tokens = len(tokenizer(sample_text).input_ids)
print(f"Sample token length: {n_tokens} (target <= {MAX_SEQ_LENGTH})")


## Cell 9 — Build HuggingFace Datasets

In [ ]:
def fmt(row):
    return format_example(row, eos_token=tokenizer.eos_token)

train_ds = Dataset.from_list(raw_train).map(
    fmt, remove_columns=["query", "output_json"]
)
valid_ds = Dataset.from_list(raw_valid).map(
    fmt, remove_columns=["query", "output_json"]
)

print(f"Train dataset : {len(train_ds)} examples")
print(f"Valid dataset : {len(valid_ds)} examples")
print("\nSample text (truncated):")
print(train_ds[0]["text"][:500])

## Cell 10 — Load Phi-2 in 4-bit (QLoRA)

Config mirrors Lab02 exactly: NF4 double-quant, bfloat16 compute dtype, gradient checkpointing.

In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Compute dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

if SKIP_TRAINING:
    # ── Inference-only path: load base model + saved adapter ──
    from peft import PeftModel

    print(f"Loading base model + saved adapter from: {BEST_CKPT_PATH}")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, BEST_CKPT_PATH)
    model.eval()
    model.config.use_cache = True
    print("Adapter loaded — model ready for inference.")
else:
    # ── Training path: load base model + prepare for k-bit training ──
    print("Loading base model for training...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    print("Base model loaded and prepared for QLoRA training.")

!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

## Cell 11 — Apply LoRA Adapters

In [ ]:
if not SKIP_TRAINING:
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "dense", "fc1", "fc2"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    # Expected: ~0.84% trainable (~23M of 2.8B params)
else:
    print("Skipping LoRA setup — adapter already loaded.")
    model.print_trainable_parameters()

## Cell 12 — Fine-Tune with SFTTrainer

Config mirrors Lab02: cosine LR, paged AdamW-8bit, effective batch 16, 2 epochs.  
Checkpoints saved to Drive so training can resume on Colab disconnect.

In [ ]:
if SKIP_TRAINING:
    print(f"Training already complete. Adapter loaded from: {BEST_CKPT_PATH}")
    print("Skipping training — proceed to inference cells below.")
else:
    import gc
    import inspect as _inspect
    import glob as _glob
    import time as _time
    from transformers import TrainerCallback

    _sft_sig = set(_inspect.signature(SFTConfig.__init__).parameters.keys())
    _trainer_sig = set(_inspect.signature(SFTTrainer.__init__).parameters.keys())

    _eval_key = "eval_strategy" if "eval_strategy" in _sft_sig else "evaluation_strategy"
    _tok_key = "processing_class" if "processing_class" in _trainer_sig else "tokenizer"

    _steps_per_epoch = max(1, len(train_ds) // (4 * 4))
    _warmup_steps = max(1, int(0.05 * _steps_per_epoch * 2))
    _total_steps = _steps_per_epoch * 2  # num_train_epochs=2

    # --- Find the highest checkpoint by step number ---
    _checkpoints = _glob.glob(f"{CKPT_DIR}/checkpoint-*")
    _resume_ckpt = None
    _training_already_done = False

    if _checkpoints:
        _checkpoints.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
        _resume_ckpt = _checkpoints[-1]
        _latest_step = int(_resume_ckpt.rsplit("-", 1)[-1])
        print(f"Found {len(_checkpoints)} checkpoint(s). Highest: step {_latest_step} / {_total_steps}")

        if _latest_step >= _total_steps:
            _training_already_done = True
            print(f"Latest checkpoint (step {_latest_step}) >= total steps ({_total_steps}).")
            print(f"Training already complete — loading adapter from: {_resume_ckpt}")
            # Reload model with trained adapter from the checkpoint
            from peft import PeftModel
            del model
            gc.collect()
            torch.cuda.empty_cache()
            _base = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
            )
            model = PeftModel.from_pretrained(_base, _resume_ckpt)
            model.eval()
            model.config.use_cache = True
            SKIP_TRAINING = True
        else:
            print(f"Resuming training from checkpoint: {_resume_ckpt} (step {_latest_step}/{_total_steps})")
    else:
        print("No checkpoint found — starting fresh.")

    if not _training_already_done:
        _cfg_extra = {}
        _trainer_extra = {}
        for _param, _val in [
            ("max_seq_length", MAX_SEQ_LENGTH),
            ("dataset_text_field", "text"),
            ("packing", False),
        ]:
            if _param in _sft_sig:
                _cfg_extra[_param] = _val
            elif _param in _trainer_sig:
                _trainer_extra[_param] = _val

        _common_args = dict(
            output_dir=CKPT_DIR,
            seed=42,
            num_train_epochs=2,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            lr_scheduler_type="cosine",
            warmup_steps=_warmup_steps,
            optim="paged_adamw_8bit",
            bf16=torch.cuda.is_bf16_supported(),
            fp16=not torch.cuda.is_bf16_supported(),
            logging_steps=50,
            save_strategy="epoch",
            save_total_limit=3,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            report_to="none",
        )

        training_args = SFTConfig(
            **_common_args,
            **{_eval_key: "epoch"},
            **_cfg_extra,
        )

        class TimedCheckpointCallback(TrainerCallback):
            """Saves adapter weights to Drive every `interval_min` minutes."""

            def __init__(self, adapter_dir, interval_min=5):
                self.adapter_dir = adapter_dir
                self.interval_sec = interval_min * 60
                self.last_save = _time.time()

            def on_step_end(self, args, state, control, model=None, **kwargs):
                elapsed = _time.time() - self.last_save
                if elapsed >= self.interval_sec:
                    save_path = f"{self.adapter_dir}/timed_ckpt_step_{state.global_step}"
                    os.makedirs(save_path, exist_ok=True)
                    model.save_pretrained(save_path)
                    tokenizer.save_pretrained(save_path)
                    self.last_save = _time.time()
                    print(f"\n[TimedCheckpoint] Saved adapter at step {state.global_step} "
                          f"to {save_path} ({elapsed/60:.1f} min since last save)")

        trainer = SFTTrainer(
            model=model,
            **{_tok_key: tokenizer},
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=valid_ds,
            callbacks=[TimedCheckpointCallback(ADAPTER_DIR, interval_min=5)],
            **_trainer_extra,
        )

        print(f"Training on {len(train_ds)} examples, validating on {len(valid_ds)}")
        print(f"Epochs: 2  |  Effective batch: 16  |  Warmup steps: {_warmup_steps}")
        print(f"Max sequence length: {MAX_SEQ_LENGTH}")
        print(f"SFT params -> SFTConfig: {_cfg_extra}  |  SFTTrainer: {_trainer_extra}")
        print(f"Tokenizer key: {_tok_key!r}")

        trainer.train(resume_from_checkpoint=_resume_ckpt)

## Cell 13 — Save Adapter Weights to Drive

In [ ]:
if not SKIP_TRAINING:
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"Adapter saved to: {ADAPTER_DIR}")
else:
    print(f"Adapter was already saved at: {BEST_CKPT_PATH}")

print("\nAdapter directory contents:")
for fname in sorted(os.listdir(ADAPTER_DIR)):
    fpath = os.path.join(ADAPTER_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {fname:<40s}  {size_mb:.2f} MB")
    else:
        print(f"  {fname:<40s}  [directory]")

## Cell 14 — Inference Function

Given a natural language question, generates the JSON output and parses it back to a Python dict.

In [ ]:
def generate_answer(question: str, max_new_tokens: int = 200) -> dict:
    """Run the fine-tuned model on a question and return a parsed JSON dict."""
    model.eval()
    model.config.use_cache = True

    prompt = make_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = out[0][inputs["input_ids"].shape[1] :]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
        return {"raw_output": raw}


## Cell 15 — Evaluation on Held-Out Queries

Runs the model on a set of test questions and cross-checks the model's answer against
the ground-truth answer computed directly by `ToolExecutor`.

In [ ]:
def normalize_answer(value):
    if isinstance(value, float):
        return round(value, 4)
    if isinstance(value, list):
        return [normalize_answer(item) for item in value]
    if isinstance(value, dict):
        return {key: normalize_answer(val) for key, val in value.items()}
    return value


test_queries = [
    {
        "question": "What is the total revenue for 2022?",
        "expected_actions": ["filter_data(column='year', value=2022)", "aggregate_sum(column='revenue')"],
    },
    {
        "question": "Which city had the highest profit in 2021? Top 1",
        "expected_actions": ["filter_data(column='year', value=2021)", "group_by(column='city')", "aggregate_sum(column='profit')", "sort_by(column='profit', order='desc')", "top_k(k=1)"],
    },
    {
        "question": "What is the average revenue by city?",
        "expected_actions": ["group_by(column='city')", "aggregate_mean(column='revenue')"],
    },
    {
        "question": "List top 3 cities by revenue in 2022.",
        "expected_actions": ["filter_data(column='year', value=2022)", "group_by(column='city')", "aggregate_sum(column='revenue')", "sort_by(column='revenue', order='desc')", "top_k(k=3)"],
    },
    {
        "question": "What is the total profit for 2023?",
        "expected_actions": ["filter_data(column='year', value=2023)", "aggregate_sum(column='profit')"],
    },
]

for entry in test_queries:
    q = entry["question"]
    expected_actions = entry["expected_actions"]
    expected_answer = compute_answer(expected_actions, df)

    result = generate_answer(q)
    model_actions = result.get("actions", [])
    model_answer = compute_answer(model_actions, df) if model_actions else None

    print(f"Q: {q}")
    print(f"  Expected actions : {expected_actions}")
    print(f"  Model actions    : {model_actions}")
    print(f"  Expected answer  : {expected_answer}")
    print(f"  Model answer     : {model_answer}")
    print(f"  Match            : {normalize_answer(model_answer) == normalize_answer(expected_answer)}")
    print()

---

# Phase 2: ToolAlpaca SFT & The ReAct Execution Loop

**Part A** — Fine-tune a 7B model on ReAct-format traces (Thought → Action → Action Input → Observation).  
**Part B** — Build a pure-Python ReAct execution engine with custom stopping criteria, robust parsing, error recovery, and bounded loop.

Uses the same `agent_trajectories_2k.json` data, but reformats it into multi-turn ReAct traces executed step-by-step against `ToolExecutor`.

## Cell 17 — Phase 2 Config & Paths

In [ ]:
# -- Phase 2 configuration --
MODEL_NAME_P2 = "mistralai/Mistral-7B-v0.1"  # 7B model as required by project spec
ADAPTER_DIR_P2 = f"{DRIVE_DIR}/mistral-react-adapter"
CKPT_DIR_P2 = f"{DRIVE_DIR}/mistral-react-qlora"
MAX_SEQ_LENGTH_P2 = 768  # ReAct traces are longer than Phase 1 JSON output

os.makedirs(ADAPTER_DIR_P2, exist_ok=True)
os.makedirs(CKPT_DIR_P2, exist_ok=True)

# Detect Phase 2 training state (same logic as Phase 1)
def detect_p2_state():
    if os.path.isfile(f"{ADAPTER_DIR_P2}/adapter_config.json"):
        return "TRAINING_COMPLETE", ADAPTER_DIR_P2
    epoch_ckpts = _glob.glob(f"{CKPT_DIR_P2}/checkpoint-*")
    if epoch_ckpts:
        epoch_ckpts.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
        return "EPOCH_CHECKPOINT", epoch_ckpts[-1]
    timed_ckpts = _glob.glob(f"{ADAPTER_DIR_P2}/timed_ckpt_step_*")
    if timed_ckpts:
        timed_ckpts.sort(key=lambda p: int(p.rsplit("_", 1)[-1]))
        return "TIMED_CHECKPOINT", timed_ckpts[-1]
    return "FRESH", None

P2_STATE, P2_CKPT_PATH = detect_p2_state()
P2_SKIP_TRAINING = P2_STATE in ("TRAINING_COMPLETE", "TIMED_CHECKPOINT")

print(f"Phase 2 model   : {MODEL_NAME_P2}")
print(f"Phase 2 state   : {P2_STATE}")
print(f"Phase 2 ckpt    : {P2_CKPT_PATH}")
print(f"Skip P2 training: {P2_SKIP_TRAINING}")

## Cell 18 — Build ReAct Training Data

Converts each trajectory into a multi-turn Thought → Action → Action Input → Observation trace by executing actions step-by-step against `ToolExecutor`.

In [ ]:
REACT_SYSTEM_PROMPT = """You are a data analysis agent. You have access to the following tools to analyze a sales dataset:

Function Descriptions:
[
  {"name": "filter_data", "description": "Filter rows where column equals value", "parameters": {"column": "str", "value": "str or int"}},
  {"name": "group_by", "description": "Group the data by a column", "parameters": {"column": "str"}},
  {"name": "aggregate_sum", "description": "Sum a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_mean", "description": "Average a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_count", "description": "Count rows for a column", "parameters": {"column": "str"}},
  {"name": "sort_by", "description": "Sort by a column", "parameters": {"column": "str", "order": "asc or desc"}},
  {"name": "top_k", "description": "Select top k rows", "parameters": {"k": "int"}}
]

Schema: date (date), year (int), month (int), city (str), region (str), product (str), category (str), revenue (float), units_sold (int), cost (float), profit (float)

Use the Thought/Action/Action Input format. Wait for Observation after each action. End with Final Answer."""

REACT_PROMPT_TEMPLATE = """### System
{system}

### User Query
{question}

### Agent Scratchpad
{scratchpad}"""


# ── Thought generation templates ──
_THOUGHT_TEMPLATES = {
    "filter_data": "I need to filter the data by {args} to narrow down the dataset.",
    "group_by": "I should group the data by {args} to organize it.",
    "aggregate_sum": "I need to compute the sum of {args}.",
    "aggregate_mean": "I need to compute the average of {args}.",
    "aggregate_count": "I need to count the entries for {args}.",
    "sort_by": "I should sort the results by {args}.",
    "top_k": "I need to select the top {args} entries.",
}


def _extract_args_text(action_str):
    """Extract human-readable args from action string."""
    m = re.search(r"\((.+)\)", action_str)
    return m.group(1) if m else action_str


def _format_observation(result_df, max_chars=500):
    """Format a DataFrame observation, truncating if too large."""
    if result_df is None:
        return "No data returned."
    if hasattr(result_df, "empty") and result_df.empty:
        return "Empty result."

    if result_df.shape == (1, 1):
        return str(result_df.iloc[0, 0])

    text = result_df.to_string(index=False)
    if len(text) > max_chars:
        text = text[:max_chars] + f"\n... (truncated, {result_df.shape[0]} rows total)"
    return text


def build_react_trace(entry, source_df):
    """Build a multi-step ReAct trace from a trajectory entry."""
    actions = entry["actions"]
    trace_lines = []
    current_df = source_df.copy()

    for i, action_str in enumerate(actions):
        action_name = action_str.split("(")[0]
        args_text = _extract_args_text(action_str)

        # Thought
        template = _THOUGHT_TEMPLATES.get(action_name, "I need to {args}.")
        thought = template.format(args=args_text)
        if i == 0:
            thought = "Let me start. " + thought

        # Action + Action Input
        # Extract just the parenthesized arguments for Action Input as JSON-like
        trace_lines.append(f"Thought: {thought}")
        trace_lines.append(f"Action: {action_name}")
        trace_lines.append(f"Action Input: {args_text}")

        # Execute to get real Observation
        parsed = parse_agent_action(action_str)
        if parsed is None:
            return None

        try:
            result = ToolExecutor(current_df).execute([parsed])
            obs = _format_observation(result)
            if isinstance(result, pd.DataFrame) and not result.empty:
                current_df = result
        except Exception as e:
            return None

        trace_lines.append(f"Observation: {obs}")

    # Final answer
    final_answer = compute_answer(entry["actions"], source_df)
    if final_answer is None:
        return None

    trace_lines.append("Thought: I now have all the information needed to answer.")
    trace_lines.append(f"Final Answer: {json.dumps(final_answer)}")

    return "\n".join(trace_lines)


# Build traces for all trajectories
react_data = []
react_skipped = 0

for entry in trajectories:
    trace = build_react_trace(entry, df)
    if trace is None:
        react_skipped += 1
        continue
    react_data.append({
        "question": entry["query"],
        "trace": trace,
    })

print(f"ReAct traces built: {len(react_data)} / {len(trajectories)}")
print(f"Skipped           : {react_skipped}")
print("\n── Sample trace ──")
print(react_data[0]["trace"][:1500])

## Cell 19 — Phase 2 Dataset & Tokenizer

In [ ]:
# Load Phase 2 tokenizer
tokenizer_p2 = AutoTokenizer.from_pretrained(MODEL_NAME_P2, trust_remote_code=True)
if tokenizer_p2.pad_token is None:
    tokenizer_p2.pad_token = tokenizer_p2.eos_token
tokenizer_p2.padding_side = "right"

EOS_P2 = tokenizer_p2.eos_token
print(f"Phase 2 EOS token: {EOS_P2!r}")


def format_react_example(row):
    """Format a ReAct trace into the full training text."""
    prompt = REACT_PROMPT_TEMPLATE.format(
        system=REACT_SYSTEM_PROMPT,
        question=row["question"],
        scratchpad="",
    )
    return {"text": prompt + row["trace"] + EOS_P2}


# Shuffle and split
random.seed(42)
random.shuffle(react_data)

P2_TRAIN_SIZE = min(1800, int(len(react_data) * 0.9))
P2_VALID_SIZE = min(200, len(react_data) - P2_TRAIN_SIZE)

p2_raw_train = react_data[:P2_TRAIN_SIZE]
p2_raw_valid = react_data[P2_TRAIN_SIZE : P2_TRAIN_SIZE + P2_VALID_SIZE]

p2_train_ds = Dataset.from_list(p2_raw_train).map(
    format_react_example, remove_columns=["question", "trace"]
)
p2_valid_ds = Dataset.from_list(p2_raw_valid).map(
    format_react_example, remove_columns=["question", "trace"]
)

# Check token lengths
sample_len = len(tokenizer_p2(p2_train_ds[0]["text"]).input_ids)
print(f"\nPhase 2 train : {len(p2_train_ds)} examples")
print(f"Phase 2 valid : {len(p2_valid_ds)} examples")
print(f"Sample token length: {sample_len} (target <= {MAX_SEQ_LENGTH_P2})")
print("\nSample text (truncated):")
print(p2_train_ds[0]["text"][:800])

## Cell 20 — Load 7B Model + LoRA (Phase 2)

Loads Mistral-7B in 4-bit QLoRA. If a trained adapter already exists, loads it directly for inference.

In [ ]:
# Free Phase 1 model memory if it's still loaded
import gc
if 'model' in dir() and model is not None:
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("Phase 1 model freed from GPU memory.")

# Phase 2 quantization config
bnb_config_p2 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

if P2_SKIP_TRAINING:
    from peft import PeftModel

    print(f"Loading base model + saved Phase 2 adapter from: {P2_CKPT_PATH}")
    base_model_p2 = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME_P2,
        quantization_config=bnb_config_p2,
        device_map="auto",
        trust_remote_code=True,
    )
    model_p2 = PeftModel.from_pretrained(base_model_p2, P2_CKPT_PATH)
    model_p2.eval()
    model_p2.config.use_cache = True
    print("Phase 2 adapter loaded — model ready for inference.")
else:
    print(f"Loading {MODEL_NAME_P2} for training...")
    model_p2 = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME_P2,
        quantization_config=bnb_config_p2,
        device_map="auto",
        trust_remote_code=True,
    )
    model_p2.config.use_cache = False
    model_p2 = prepare_model_for_kbit_training(model_p2, use_gradient_checkpointing=True)

    # LoRA config for Mistral-7B
    lora_config_p2 = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model_p2 = get_peft_model(model_p2, lora_config_p2)
    model_p2.print_trainable_parameters()

!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

## Cell 21 — Phase 2 SFT Training

Fine-tunes the 7B model on ReAct-format traces. Uses the same checkpoint resume logic as Phase 1.

In [ ]:
if P2_SKIP_TRAINING:
    print(f"Phase 2 training already complete. Adapter loaded from: {P2_CKPT_PATH}")
    print("Skipping training — proceed to ReAct execution cells below.")
else:
    import gc
    import inspect as _inspect
    import glob as _glob
    import time as _time
    from transformers import TrainerCallback

    _sft_sig_p2 = set(_inspect.signature(SFTConfig.__init__).parameters.keys())
    _trainer_sig_p2 = set(_inspect.signature(SFTTrainer.__init__).parameters.keys())

    _eval_key_p2 = "eval_strategy" if "eval_strategy" in _sft_sig_p2 else "evaluation_strategy"
    _tok_key_p2 = "processing_class" if "processing_class" in _trainer_sig_p2 else "tokenizer"

    # Smaller batch for 7B model on T4
    _p2_batch = 2
    _p2_grad_accum = 8  # effective batch = 16
    _steps_per_epoch_p2 = max(1, len(p2_train_ds) // (_p2_batch * _p2_grad_accum))
    _warmup_steps_p2 = max(1, int(0.05 * _steps_per_epoch_p2 * 2))
    _total_steps_p2 = _steps_per_epoch_p2 * 2  # num_train_epochs=2

    # --- Find the highest checkpoint by step number ---
    _ckpts_p2 = _glob.glob(f"{CKPT_DIR_P2}/checkpoint-*")
    _resume_p2 = None
    _p2_training_already_done = False

    if _ckpts_p2:
        _ckpts_p2.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
        _resume_p2 = _ckpts_p2[-1]
        _latest_step_p2 = int(_resume_p2.rsplit("-", 1)[-1])
        print(f"Found {len(_ckpts_p2)} checkpoint(s). Highest: step {_latest_step_p2} / {_total_steps_p2}")

        if _latest_step_p2 >= _total_steps_p2:
            _p2_training_already_done = True
            print(f"Latest checkpoint (step {_latest_step_p2}) >= total steps ({_total_steps_p2}).")
            print(f"Phase 2 training already complete — loading adapter from: {_resume_p2}")
            # Reload model with trained adapter from the checkpoint
            from peft import PeftModel
            del model_p2
            gc.collect()
            torch.cuda.empty_cache()
            _base_p2 = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME_P2,
                quantization_config=bnb_config_p2,
                device_map="auto",
                trust_remote_code=True,
            )
            model_p2 = PeftModel.from_pretrained(_base_p2, _resume_p2)
            model_p2.eval()
            model_p2.config.use_cache = True
            P2_SKIP_TRAINING = True
        else:
            print(f"Resuming Phase 2 training from checkpoint: {_resume_p2} (step {_latest_step_p2}/{_total_steps_p2})")
    else:
        print("No Phase 2 checkpoint found — starting fresh.")

    if not _p2_training_already_done:
        _cfg_extra_p2 = {}
        _trainer_extra_p2 = {}
        for _param, _val in [
            ("max_seq_length", MAX_SEQ_LENGTH_P2),
            ("dataset_text_field", "text"),
            ("packing", False),
        ]:
            if _param in _sft_sig_p2:
                _cfg_extra_p2[_param] = _val
            elif _param in _trainer_sig_p2:
                _trainer_extra_p2[_param] = _val

        _common_args_p2 = dict(
            output_dir=CKPT_DIR_P2,
            seed=42,
            num_train_epochs=2,
            per_device_train_batch_size=_p2_batch,
            gradient_accumulation_steps=_p2_grad_accum,
            learning_rate=2e-4,
            lr_scheduler_type="cosine",
            warmup_steps=_warmup_steps_p2,
            optim="paged_adamw_8bit",
            bf16=torch.cuda.is_bf16_supported(),
            fp16=not torch.cuda.is_bf16_supported(),
            logging_steps=25,
            save_strategy="epoch",
            save_total_limit=3,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            report_to="none",
        )

        training_args_p2 = SFTConfig(
            **_common_args_p2,
            **{_eval_key_p2: "epoch"},
            **_cfg_extra_p2,
        )

        class TimedCheckpointCallbackP2(TrainerCallback):
            """Saves adapter weights to Drive every `interval_min` minutes."""

            def __init__(self, adapter_dir, interval_min=5):
                self.adapter_dir = adapter_dir
                self.interval_sec = interval_min * 60
                self.last_save = _time.time()

            def on_step_end(self, args, state, control, model=None, **kwargs):
                elapsed = _time.time() - self.last_save
                if elapsed >= self.interval_sec:
                    save_path = f"{self.adapter_dir}/timed_ckpt_step_{state.global_step}"
                    os.makedirs(save_path, exist_ok=True)
                    model.save_pretrained(save_path)
                    tokenizer_p2.save_pretrained(save_path)
                    self.last_save = _time.time()
                    print(f"\n[P2 TimedCheckpoint] Saved at step {state.global_step} "
                          f"to {save_path} ({elapsed/60:.1f} min)")

        trainer_p2 = SFTTrainer(
            model=model_p2,
            **{_tok_key_p2: tokenizer_p2},
            args=training_args_p2,
            train_dataset=p2_train_ds,
            eval_dataset=p2_valid_ds,
            callbacks=[TimedCheckpointCallbackP2(ADAPTER_DIR_P2, interval_min=5)],
            **_trainer_extra_p2,
        )

        print(f"Phase 2 training: {len(p2_train_ds)} examples, validating on {len(p2_valid_ds)}")
        print(f"Epochs: 2 | Batch: {_p2_batch} | Grad accum: {_p2_grad_accum} | Warmup: {_warmup_steps_p2}")
        print(f"Max sequence length: {MAX_SEQ_LENGTH_P2}")

        trainer_p2.train(resume_from_checkpoint=_resume_p2)

## Cell 22 — Save Phase 2 Adapter

In [ ]:
if not P2_SKIP_TRAINING:
    model_p2.save_pretrained(ADAPTER_DIR_P2)
    tokenizer_p2.save_pretrained(ADAPTER_DIR_P2)
    print(f"Phase 2 adapter saved to: {ADAPTER_DIR_P2}")
else:
    print(f"Phase 2 adapter was already saved at: {P2_CKPT_PATH}")

print("\nPhase 2 adapter directory contents:")
for fname in sorted(os.listdir(ADAPTER_DIR_P2)):
    fpath = os.path.join(ADAPTER_DIR_P2, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {fname:<40s}  {size_mb:.2f} MB")
    else:
        print(f"  {fname:<40s}  [directory]")

## Cell 23 — ReAct Execution Engine (Part B)

Pure Python implementation of the ReAct agent loop. No LangChain/LlamaIndex.

Key features:
1. **Custom StoppingCriteria** — halts generation at "Observation:" to prevent hallucinated tool outputs
2. **Robust JSON parsing** — sanitizes malformed JSON (trailing commas, markdown blocks)
3. **State management** — truncates large observations to prevent OOM
4. **Bounded error recovery** — max 2 consecutive errors before graceful termination
5. **Final Answer detection** — breaks loop when model outputs "Final Answer:"

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList


class ObservationStopCriteria(StoppingCriteria):
    """Halt generation when the model produces 'Observation:' or 'Final Answer:'.

    We check the decoded tail of the generated sequence for these trigger strings,
    which is more robust than matching exact token IDs across tokenizers.
    """

    def __init__(self, tokenizer, triggers=("Observation:", "Final Answer:")):
        self.tokenizer = tokenizer
        self.triggers = triggers

    def __call__(self, input_ids, scores, **kwargs):
        # Decode the last 30 tokens to check for triggers
        tail = self.tokenizer.decode(input_ids[0, -30:], skip_special_tokens=True)
        return any(trigger in tail for trigger in self.triggers)


def _sanitize_json(text):
    """Best-effort cleanup for malformed JSON from smaller models."""
    # Strip markdown code fences
    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```\s*", "", text)
    # Remove trailing commas before } or ]
    text = re.sub(r",\s*([}\]])", r"\1", text)
    return text.strip()


class ReActAgent:
    """Pure-Python ReAct execution engine for the fine-tuned 7B model.

    Implements a bounded while-loop that:
    - Generates model output with custom stopping criteria
    - Parses Thought/Action/Action Input from the output
    - Executes the action via ToolExecutor
    - Appends the real Observation to the scratchpad
    - Repeats until Final Answer is detected or max_steps is reached
    """

    def __init__(self, model, tokenizer, source_df, max_steps=5, max_obs_chars=500,
                 max_consecutive_errors=2, max_new_tokens=256):
        self.model = model
        self.tokenizer = tokenizer
        self.source_df = source_df
        self.max_steps = max_steps
        self.max_obs_chars = max_obs_chars
        self.max_consecutive_errors = max_consecutive_errors
        self.max_new_tokens = max_new_tokens

        self.stop_criteria = StoppingCriteriaList([
            ObservationStopCriteria(tokenizer)
        ])

    def run(self, question):
        """Execute the ReAct loop for a given question. Returns a dict with the result."""
        self.model.eval()
        self.model.config.use_cache = True

        current_df = self.source_df.copy()
        scratchpad = ""
        consecutive_errors = 0
        all_actions = []

        for step in range(self.max_steps):
            # Build the full prompt
            prompt = REACT_PROMPT_TEMPLATE.format(
                system=REACT_SYSTEM_PROMPT,
                question=question,
                scratchpad=scratchpad,
            )

            # Generate
            output_text = self._generate(prompt)
            scratchpad += output_text

            # Check for Final Answer
            if "Final Answer:" in output_text:
                answer_text = output_text.split("Final Answer:")[-1].strip()
                # Remove EOS tokens
                answer_text = answer_text.replace(self.tokenizer.eos_token, "").strip()
                return {
                    "actions": all_actions,
                    "answer": self._parse_final_answer(answer_text),
                    "steps": step + 1,
                    "scratchpad": scratchpad,
                }

            # Parse Action and Action Input
            action_name, action_input = self._parse_action(output_text)

            if action_name is None:
                consecutive_errors += 1
                error_msg = "Could not parse action from output. Use format: Action: <name>\\nAction Input: <args>"
                scratchpad += f"\nObservation: ERROR — {error_msg}\n"
                if consecutive_errors >= self.max_consecutive_errors:
                    return {
                        "actions": all_actions,
                        "answer": None,
                        "error": f"Max consecutive errors ({self.max_consecutive_errors}) reached.",
                        "steps": step + 1,
                        "scratchpad": scratchpad,
                    }
                continue

            # Reset error counter on successful parse
            consecutive_errors = 0

            # Reconstruct the action string and execute
            action_str = f"{action_name}({action_input})"
            all_actions.append(action_str)

            try:
                parsed = parse_agent_action(action_str)
                if parsed is None:
                    raise ValueError(f"Unknown action: {action_str}")
                result = ToolExecutor(current_df).execute([parsed])
                obs = _format_observation(result, max_chars=self.max_obs_chars)
                if isinstance(result, pd.DataFrame) and not result.empty:
                    current_df = result
            except Exception as e:
                obs = f"ERROR — {type(e).__name__}: {str(e)}"

            scratchpad += f"\nObservation: {obs}\n"

        # Max steps exhausted
        return {
            "actions": all_actions,
            "answer": None,
            "error": f"Max steps ({self.max_steps}) reached without Final Answer.",
            "steps": self.max_steps,
            "scratchpad": scratchpad,
        }

    def _generate(self, prompt):
        """Generate text from the model, stopping at Observation: or Final Answer:."""
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True,
                                max_length=MAX_SEQ_LENGTH_P2).to(self.model.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                stopping_criteria=self.stop_criteria,
            )

        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    def _parse_action(self, text):
        """Extract Action and Action Input from generated text."""
        action_match = re.search(r"Action:\s*(\w+)", text)
        input_match = re.search(r"Action Input:\s*(.+?)(?:\n|$)", text, re.DOTALL)

        if action_match and input_match:
            return action_match.group(1).strip(), input_match.group(1).strip()
        return None, None

    def _parse_final_answer(self, text):
        """Robustly parse the Final Answer text."""
        text = _sanitize_json(text)

        # Try JSON first
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass

        # Try extracting a JSON object/array
        for pattern in [r"\{[^{}]*\}", r"\[.*\]"]:
            match = re.search(pattern, text, re.DOTALL)
            if match:
                try:
                    return json.loads(_sanitize_json(match.group()))
                except json.JSONDecodeError:
                    pass

        # Try plain number
        num_match = re.search(r"-?[\d]+(?:\.[\d]+)?", text)
        if num_match:
            val = num_match.group()
            return float(val) if '.' in val else int(val)

        return text


print("ReActAgent class defined.")
print("ObservationStopCriteria: halts at 'Observation:' and 'Final Answer:'")
print("Ready for inference.")

## Cell 24 — ReAct Agent Demo & Evaluation

Runs the ReAct agent on test queries, showing the full multi-turn trace and comparing against ground truth.

In [ ]:
agent = ReActAgent(model_p2, tokenizer_p2, df, max_steps=5)

react_test_queries = [
    {
        "question": "What is the total revenue for 2022?",
        "expected_actions": ["filter_data(column='year', value=2022)", "aggregate_sum(column='revenue')"],
    },
    {
        "question": "Which city had the highest profit in 2021? Top 1",
        "expected_actions": ["filter_data(column='year', value=2021)", "group_by(column='city')", "aggregate_sum(column='profit')", "sort_by(column='profit', order='desc')", "top_k(k=1)"],
    },
    {
        "question": "What is the average revenue by city?",
        "expected_actions": ["group_by(column='city')", "aggregate_mean(column='revenue')"],
    },
    {
        "question": "List top 3 cities by revenue in 2022.",
        "expected_actions": ["filter_data(column='year', value=2022)", "group_by(column='city')", "aggregate_sum(column='revenue')", "sort_by(column='revenue', order='desc')", "top_k(k=3)"],
    },
    {
        "question": "What is the total profit for 2023?",
        "expected_actions": ["filter_data(column='year', value=2023)", "aggregate_sum(column='profit')"],
    },
]

print("=" * 70)
print("PHASE 2 -- ReAct Agent Evaluation")
print("=" * 70)

for entry in react_test_queries:
    q = entry["question"]
    expected_actions = entry["expected_actions"]
    expected_answer = compute_answer(expected_actions, df)

    result = agent.run(q)
    model_actions = result.get("actions", [])
    model_answer = compute_answer(model_actions, df) if model_actions else None
    steps = result.get("steps", "?")
    error = result.get("error")

    print(f"\nQ: {q}")
    print(f"  Steps            : {steps}")
    print(f"  Expected actions : {expected_actions}")
    print(f"  Model actions    : {model_actions}")
    print(f"  Expected answer  : {expected_answer}")
    print(f"  Model answer     : {model_answer}")
    if error:
        print(f"  Error            : {error}")
    print(f"  Match            : {normalize_answer(model_answer) == normalize_answer(expected_answer)}")
    print("-" * 70)

    # Show full scratchpad for the first query
    if q == react_test_queries[0]["question"]:
        print("\n  -- Full Scratchpad (first query) --")
        print(result.get("scratchpad", "")[:2000])
        print("  -- End Scratchpad --")
        print("-" * 70)

## Cell 25 — (Bonus) Direct Preference Optimization (DPO)

DPO training on top of the SFT model. Creates preference pairs:
- **Chosen**: Correct ReAct traces (from training data)
- **Rejected**: Perturbed traces (wrong column names, wrong action order, hallucinated outputs)

Uses `trl.DPOTrainer` with the SFT model as the reference model.

In [ ]:
# -- Bonus: DPO Training --
# This cell creates preference pairs and runs DPO on the Phase 2 SFT model.

DPO_ADAPTER_DIR = f"{DRIVE_DIR}/mistral-react-dpo-adapter"
DPO_CKPT_DIR = f"{DRIVE_DIR}/mistral-react-dpo-ckpt"
os.makedirs(DPO_ADAPTER_DIR, exist_ok=True)
os.makedirs(DPO_CKPT_DIR, exist_ok=True)

# -- Step 1: Generate rejected traces by perturbing correct ones --

WRONG_COLUMNS = ["invalid_col", "nonexistent", "revenue2", "total", "amount"]
VALID_COLUMNS = ["date", "year", "month", "city", "region", "product", "category",
                 "revenue", "units_sold", "cost", "profit"]


def perturb_trace(trace_text):
    """Create a rejected version of a ReAct trace by introducing errors."""
    lines = trace_text.split("\n")
    perturbed = []
    changed = False

    for line in lines:
        if line.startswith("Action Input:") and random.random() < 0.5:
            # Replace a valid column with a wrong one.
            for col in VALID_COLUMNS:
                if col in line:
                    wrong = random.choice(WRONG_COLUMNS)
                    line = line.replace(col, wrong, 1)
                    changed = True
                    break
        elif line.startswith("Thought:") and random.random() < 0.3:
            # Add conversational filler; the trained model should avoid this.
            line = "Thought: Sure! I'd be happy to help with that! " + line[9:]
            changed = True
        perturbed.append(line)

    # Ensure the rejected trace is different from the chosen trace.
    if not changed:
        for i, line in enumerate(perturbed):
            if line.startswith("Action Input:"):
                for col in VALID_COLUMNS:
                    if col in line:
                        perturbed[i] = line.replace(col, WRONG_COLUMNS[0], 1)
                        changed = True
                        break
                if changed:
                    break

    if not changed:
        for i, line in enumerate(perturbed):
            if line.startswith("Thought:"):
                perturbed[i] = "Thought: Sure! I'd be happy to help with that! " + line[9:]
                break

    return "\n".join(perturbed)


# Build DPO dataset
dpo_data = []
for item in react_data[:500]:  # Use a subset for DPO
    prompt = REACT_PROMPT_TEMPLATE.format(
        system=REACT_SYSTEM_PROMPT,
        question=item["question"],
        scratchpad="",
    )
    chosen = item["trace"]
    rejected = perturb_trace(item["trace"])

    dpo_data.append({
        "prompt": prompt,
        "chosen": chosen + EOS_P2,
        "rejected": rejected + EOS_P2,
    })

dpo_ds = Dataset.from_list(dpo_data)
print(f"DPO preference pairs: {len(dpo_ds)}")
print("\n-- Sample chosen (first 300 chars) --")
print(dpo_ds[0]["chosen"][:300])
print("\n-- Sample rejected (first 300 chars) --")
print(dpo_ds[0]["rejected"][:300])


# -- Step 2: DPO Training --
import gc
import inspect as _inspect

from peft import PeftModel
from trl import DPOConfig, DPOTrainer

_dpo_cfg_sig = set(_inspect.signature(DPOConfig.__init__).parameters.keys())
_dpo_trainer_sig = set(_inspect.signature(DPOTrainer.__init__).parameters.keys())
_dpo_tok_key = "processing_class" if "processing_class" in _dpo_trainer_sig else "tokenizer"

_dpo_cfg_extra = {}
_dpo_trainer_extra = {}
for _param, _value in [
    ("max_length", MAX_SEQ_LENGTH_P2),
    ("max_prompt_length", MAX_SEQ_LENGTH_P2 // 2),
    ("max_completion_length", MAX_SEQ_LENGTH_P2 // 2),
]:
    if _param in _dpo_cfg_sig:
        _dpo_cfg_extra[_param] = _value
    elif _param in _dpo_trainer_sig:
        _dpo_trainer_extra[_param] = _value

# If Phase 2 was loaded from an existing adapter, reload it in trainable mode for DPO.
if P2_SKIP_TRAINING:
    print(f"Reloading Phase 2 adapter for DPO from: {P2_CKPT_PATH}")
    del model_p2
    gc.collect()
    torch.cuda.empty_cache()

    _base_model_dpo = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME_P2,
        quantization_config=bnb_config_p2,
        device_map="auto",
        trust_remote_code=True,
    )
    _base_model_dpo.config.use_cache = False
    _base_model_dpo = prepare_model_for_kbit_training(
        _base_model_dpo, use_gradient_checkpointing=True
    )
    model_p2 = PeftModel.from_pretrained(
        _base_model_dpo,
        P2_CKPT_PATH,
        is_trainable=True,
    )
else:
    model_p2.config.use_cache = False

model_p2.train()
if hasattr(model_p2, "print_trainable_parameters"):
    model_p2.print_trainable_parameters()

dpo_config = DPOConfig(
    output_dir=DPO_CKPT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="paged_adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    beta=0.1,
    **_dpo_cfg_extra,
)

dpo_trainer = DPOTrainer(
    model=model_p2,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_ds,
    **{_dpo_tok_key: tokenizer_p2},
    **_dpo_trainer_extra,
)

print(f"DPO tokenizer key: {_dpo_tok_key!r}")
print(f"DPO params -> DPOConfig: {_dpo_cfg_extra} | DPOTrainer: {_dpo_trainer_extra}")
print("Starting DPO training...")
dpo_trainer.train()

# Save DPO adapter
model_p2.save_pretrained(DPO_ADAPTER_DIR)
tokenizer_p2.save_pretrained(DPO_ADAPTER_DIR)
print(f"\nDPO adapter saved to: {DPO_ADAPTER_DIR}")